# KarawangPadiGuard - Feature Engineering

Notebook ini untuk feature engineering pada data cuaca dan satelit untuk prediksi risiko penyakit padi.

## Penyakit yang Diprediksi:
1. **Blast** (Pyricularia oryzae) - Dipengaruhi suhu 25-28°C dan kelembapan >90%
2. **Wereng Coklat** (Nilaparvata lugens) - Dipengaruhi suhu 25-30°C dan kelembapan 80-90%
3. **Bercak Coklat** (Helminthosporium oryzae) - Dipengaruhi suhu 28-32°C dan kelembapan >85%

In [ ]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seed
np.random.seed(42)

print("✅ Libraries imported successfully")

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

# Paths
BASE_DIR = Path().absolute().parent
DATA_DIR = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR = BASE_DIR / 'data' / 'features'
MODEL_DIR = BASE_DIR / 'data' / 'splits'

# Create directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Karawang coordinates
KARAWANG_CENTER = {'lat': -6.32, 'lon': 107.29}

# Disease thresholds
DISEASE_THRESHOLDS = {
    'blast': {
        'temp_min': 25,
        'temp_max': 28,
        'humidity_min': 90
    },
    'wereng': {
        'temp_min': 25,
        'temp_max': 30,
        'humidity_min': 80,
        'humidity_max': 90
    },
    'bercak': {
        'temp_min': 28,
        'temp_max': 32,
        'humidity_min': 85
    }
}

print(f"📁 Base directory: {BASE_DIR}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")

## 1. Load Datasets

In [ ]:
# ============================================
# LOAD DATASETS
# ============================================

print("="*70)
print("LOADING DATASETS")
print("="*70)

# Load weather data
weather_path = DATA_DIR / 'weather_data.csv'
if weather_path.exists():
    weather_df = pd.read_csv(weather_path)
    weather_df['date'] = pd.to_datetime(weather_df['date'])
    print(f"\n✅ Weather data loaded: {len(weather_df):,} records")
    print(f"   Date range: {weather_df['date'].min()} to {weather_df['date'].max()}")
    print(f"   Columns: {list(weather_df.columns)}")
else:
    print(f"❌ Weather data not found at: {weather_path}")
    weather_df = None

# Load satellite data
satellite_path = DATA_DIR / 'satellite_data.csv'
if satellite_path.exists():
    satellite_df = pd.read_csv(satellite_path)
    satellite_df['date'] = pd.to_datetime(satellite_df['date'])
    print(f"\n✅ Satellite data loaded: {len(satellite_df):,} records")
    print(f"   Date range: {satellite_df['date'].min()} to {satellite_df['date'].max()}")
    print(f"   Columns: {list(satellite_df.columns)}")
else:
    print(f"❌ Satellite data not found at: {satellite_path}")
    satellite_df = None

print("\n" + "="*70)

## 2. Merge Datasets

In [ ]:
# ============================================
# MERGE DATASETS
# ============================================

if weather_df is not None and satellite_df is not None:
    print("="*70)
    print("MERGING DATASETS")
    print("="*70)
    
    # Aggregate weather data by date (average across stations)
    weather_daily = weather_df.groupby('date').agg({
        'temperature': 'mean',
        'humidity': 'mean',
        'rainfall': 'sum',
        'wind_speed': 'mean',
        'cloud_cover': 'mean',
        'pressure': 'mean',
        'dew_point': 'mean'
    }).reset_index()
    
    print(f"\n📊 Weather data aggregated: {len(weather_daily)} daily records")
    print(f"📊 Satellite data: {len(satellite_df)} weekly records")
    
    # Merge with satellite data (left join on date)
    df = weather_daily.merge(satellite_df, on='date', how='left')
    
    # Forward fill satellite data (weekly to daily)
    satellite_cols = ['ndvi', 'ndwi', 'evi', 'savi', 'anomaly']
    df[satellite_cols] = df[satellite_cols].fillna(method='ffill')
    
    # Backward fill remaining NaN values
    df[satellite_cols] = df[satellite_cols].fillna(method='bfill')
    
    # Drop any remaining rows with NaN
    df = df.dropna()
    
    print(f"\n✅ Datasets merged: {len(df)} daily records")
    print(f"   Columns after merge: {list(df.columns)}")
    
    # Display sample
    print(f"\n📋 Sample data:")
    print(df.head(10))
    
else:
    print("❌ Cannot merge - one or both datasets missing")
    df = weather_df if weather_df is not None else satellite_df

## 3. Temporal Features

In [ ]:
# ============================================
# TEMPORAL FEATURES
# ============================================

print("="*70)
print("CREATING TEMPORAL FEATURES")
print("="*70)

# Basic temporal features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek
df['day_of_year'] = df['date'].dt.dayofyear
df['week'] = df['date'].dt.isocalendar().week.astype(int)
df['quarter'] = df['date'].dt.quarter

# Seasonality features (cyclical encoding)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

# Season indicator (based on month)
df['is_rainy_season'] = df['month'].isin([11, 12, 1, 2, 3, 4]).astype(int)
df['is_dry_season'] = df['month'].isin([5, 6, 7, 8, 9, 10]).astype(int)

# Planting season indicator
# Musim tanam padi: biasanya Nov-Mar (MT1) dan Apr-Ag (MT2)
df['is_planting_season'] = df['month'].isin([11, 12, 1, 2, 3, 4, 5, 6, 7, 8]).astype(int)

# Harvest season indicator
df['is_harvest_season'] = df['month'].isin([3, 4, 9, 10]).astype(int)

print(f"\n✅ Temporal features created")
print(f"   Total features: {len(df.columns)}")
print(f"   New features: year, month, day, day_of_week, day_of_year, week, quarter")
print(f"                 month_sin, month_cos, day_of_year_sin, day_of_year_cos")
print(f"                 is_rainy_season, is_dry_season, is_planting_season, is_harvest_season")

## 4. Lag Features

In [ ]:
# ============================================
# LAG FEATURES
# ============================================

print("="*70)
print("CREATING LAG FEATURES")
print("="*70)

# Weather lag features
weather_cols = ['temperature', 'humidity', 'rainfall', 'wind_speed']

for col in weather_cols:
    for lag in [1, 3, 7]:
        df[f'{col}_lag{lag}'] = df[col].shift(lag)

# Satellite lag features
satellite_cols = ['ndvi', 'ndwi', 'evi', 'savi']

for col in satellite_cols:
    for lag in [1, 2, 4]:  # Weekly lags
        df[f'{col}_lag{lag}'] = df[col].shift(lag * 7) if lag > 1 else df[col].shift(7)

# Disease indicator lag (previous week conditions)
df['disease_pressure_lag7'] = df[
    (df['temperature'].between(25, 30)) & 
    (df['humidity'] > 80)
].astype(int).shift(7).fillna(0)

# Drop rows with NaN from lag features
df = df.dropna()

print(f"\n✅ Lag features created")
print(f"   Weather lags: 1, 3, 7 days for temp, humidity, rainfall, wind_speed")
print(f"   Satellite lags: 1, 2, 4 weeks for NDVI, NDWI, EVI, SAVI")
print(f"   Records after dropping NaN: {len(df)}")

## 5. Rolling Statistics

In [ ]:
# ============================================
# ROLLING STATISTICS
# ============================================

print("="*70)
print("CREATING ROLLING STATISTICS")
print("="*70)

# Weather rolling statistics
for col in ['temperature', 'humidity', 'rainfall']:
    # Mean
    df[f'{col}_rolling3_mean'] = df[col].rolling(window=3).mean()
    df[f'{col}_rolling7_mean'] = df[col].rolling(window=7).mean()
    df[f'{col}_rolling14_mean'] = df[col].rolling(window=14).mean()
    
    # Std
    df[f'{col}_rolling7_std'] = df[col].rolling(window=7).std()
    df[f'{col}_rolling14_std'] = df[col].rolling(window=14).std()
    
    # Min/Max
    df[f'{col}_rolling7_min'] = df[col].rolling(window=7).min()
    df[f'{col}_rolling7_max'] = df[col].rolling(window=7).max()

# Rainfall cumulative
df['rainfall_cumsum_7d'] = df['rainfall'].rolling(window=7).sum()
df['rainfall_cumsum_14d'] = df['rainfall'].rolling(window=14).sum()
df['rainfall_cumsum_30d'] = df['rainfall'].rolling(window=30).sum()

# Temperature range (diurnal variation proxy)
df['temp_range_7d'] = df['temperature_rolling7_max'] - df['temperature_rolling7_min']

# Satellite rolling statistics
for col in ['ndvi', 'ndwi', 'evi']:
    df[f'{col}_rolling2_mean'] = df[col].rolling(window=14).mean()  # 2 weeks
    df[f'{col}_rolling4_mean'] = df[col].rolling(window=28).mean()  # 4 weeks
    
    # Rate of change
    df[f'{col}_rate_of_change'] = df[col].diff(7)

# Drop rows with NaN from rolling features
df = df.dropna()

print(f"\n✅ Rolling statistics created")
print(f"   Weather: 3/7/14-day mean, 7/14-day std, 7-day min/max")
print(f"   Rainfall: 7/14/30-day cumulative sum")
print(f"   Satellite: 2/4-week mean, rate of change")
print(f"   Records after dropping NaN: {len(df)}")

## 6. Derived Features

In [ ]:
# ============================================
# DERIVED FEATURES
# ============================================

print("="*70)
print("CREATING DERIVED FEATURES")
print("="*70)

# Heat index (feels like temperature)
df['heat_index'] = df['temperature'] + 0.55 * (1 - df['humidity'] / 100) * (df['temperature'] - 14.5)

# Dew point depression
df['dew_point_depression'] = df['temperature'] - df['dew_point']

# Vegetation health indicators
df['vegetation_stress'] = (df['ndvi'] < 0.4).astype(int)
df['water_stress'] = (df['ndwi'] < 0).astype(int)
df['healthy_vegetation'] = (df['ndvi'] >= 0.6).astype(int)

# Vegetation decline indicator
df['ndvi_declining'] = (df['ndvi_rate_of_change'] < -0.05).astype(int)

# Extreme weather indicators
df['extreme_hot'] = (df['temperature'] > 35).astype(int)
df['heavy_rain'] = (df['rainfall'] > 50).astype(int)
df['high_humidity'] = (df['humidity'] > 90).astype(int)
df['low_humidity'] = (df['humidity'] < 60).astype(int)

# Disease favorable conditions
df['blast_favorable'] = (
    (df['temperature'].between(DISEASE_THRESHOLDS['blast']['temp_min'], DISEASE_THRESHOLDS['blast']['temp_max'])) &
    (df['humidity'] >= DISEASE_THRESHOLDS['blast']['humidity_min'])
).astype(int)

df['wereng_favorable'] = (
    (df['temperature'].between(DISEASE_THRESHOLDS['wereng']['temp_min'], DISEASE_THRESHOLDS['wereng']['temp_max'])) &
    (df['humidity'].between(DISEASE_THRESHOLDS['wereng']['humidity_min'], DISEASE_THRESHOLDS['wereng']['humidity_max']))
).astype(int)

df['bercak_favorable'] = (
    (df['temperature'].between(DISEASE_THRESHOLDS['bercak']['temp_min'], DISEASE_THRESHOLDS['bercak']['temp_max'])) &
    (df['humidity'] >= DISEASE_THRESHOLDS['bercak']['humidity_min'])
).astype(int)

# Combined disease risk index
df['disease_risk_index'] = (
    df['blast_favorable'] * 0.35 +
    df['wereng_favorable'] * 0.35 +
    df['bercak_favorable'] * 0.30
)

# Stress compound index (weather + vegetation)
df['stress_index'] = (
    df['vegetation_stress'] * 0.4 +
    df['water_stress'] * 0.3 +
    df['dew_point_depression'] / 10 * 0.3
)

print(f"\n✅ Derived features created")
print(f"   Weather: heat_index, dew_point_depression")
print(f"   Vegetation: stress, water_stress, healthy, declining")
print(f"   Extreme: hot, heavy_rain, high_humidity, low_humidity")
print(f"   Disease: blast_favorable, wereng_favorable, bercak_favorable")
print(f"   Indexes: disease_risk_index, stress_index")

## 7. Target Variable Generation

In [ ]:
# ============================================
# TARGET VARIABLE GENERATION
# ============================================

print("="*70)
print("GENERATING TARGET VARIABLES")
print("="*70)

# For demo purposes, generate synthetic target based on features
# In production, this would come from actual disease incidence data

# Base probability from conditions
df['disease_probability'] = (
    df['disease_risk_index'] * 0.5 +
    df['stress_index'] * 0.3 +
    (1 - df['ndvi'].clip(0, 1)) * 0.2  # Lower NDVI = higher risk
)

# Add some noise for realism
df['disease_probability'] = df['disease_probability'] + np.random.normal(0, 0.05, len(df))
df['disease_probability'] = df['disease_probability'].clip(0, 1)

# Binary outbreak indicator (probability > 0.6)
df['outbreak'] = (df['disease_probability'] > 0.6).astype(int)

# Risk level categories
def get_risk_level(prob):
    if prob < 0.3:
        return 'LOW'
    elif prob < 0.6:
        return 'MEDIUM'
    else:
        return 'HIGH'

df['risk_level'] = df['disease_probability'].apply(get_risk_level)

# Disease type (multiclass)
def get_disease_type(row):
    if row['outbreak'] == 0:
        return 'HEALTHY'
    elif row['blast_favorable'] == 1:
        return 'BLAST'
    elif row['wereng_favorable'] == 1:
        return 'WERENG'
    elif row['bercak_favorable'] == 1:
        return 'BERCAK'
    else:
        return 'OTHER'

df['disease_type'] = df.apply(get_disease_type, axis=1)

# Display target distribution
print(f"\n📊 Target Variable Distribution:")
print(f"   Outbreak rate: {df['outbreak'].mean()*100:.1f}%")
print(f"   Risk levels:")
for level in ['LOW', 'MEDIUM', 'HIGH']:
    count = (df['risk_level'] == level).sum()
    pct = count / len(df) * 100
    print(f"      {level}: {count} ({pct:.1f}%)")

print(f"\n   Disease types:")
for dtype in ['HEALTHY', 'BLAST', 'WERENG', 'BERCAK', 'OTHER']:
    count = (df['disease_type'] == dtype).sum()
    pct = count / len(df) * 100
    print(f"      {dtype}: {count} ({pct:.1f}%)")

print(f"\n✅ Target variables generated")

## 8. Feature Selection & Importance

In [ ]:
# ============================================
# FEATURE IMPORTANCE ANALYSIS
# ============================================

print("="*70)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*70)

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Select feature columns (exclude target and date)
exclude_cols = ['date', 'disease_probability', 'outbreak', 'risk_level', 'disease_type', 'month']
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Encode categorical features
le = LabelEncoder()
y = le.fit_transform(df['disease_type'])

# Prepare features
X = df[feature_cols].copy()

# Handle any remaining categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X[col] = pd.to_numeric(X[col], errors='coerce')

X = X.fillna(0)

# Train Random Forest for importance
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Get feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

# Display top features
print(f"\n📊 Top 20 Most Important Features:")
print(importance_df.head(20).to_string(index=False))

# Select top features
top_n = 40
selected_features = importance_df.head(top_n)['feature'].tolist()
print(f"\n✅ Selected {top_n} features for model training")

## 9. Train-Val-Test Split

In [ ]:
# ============================================
# TIME-SERIES AWARE TRAIN-VAL-TEST SPLIT
# ============================================

print("="*70)
print("CREATING TRAIN-VAL-TEST SPLITS")
print("="*70)

# Sort by date
df = df.sort_values('date').reset_index(drop=True)

# Calculate split indices
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Create splits
train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(f"\n📊 Split Summary:")
print(f"   Train:      {len(train_df):,} records ({len(train_df)/n*100:.1f}%)")
print(f"              {train_df['date'].min()} to {train_df['date'].max()}")
print(f"   Validation: {len(val_df):,} records ({len(val_df)/n*100:.1f}%)")
print(f"              {val_df['date'].min()} to {val_df['date'].max()}")
print(f"   Test:       {len(test_df):,} records ({len(test_df)/n*100:.1f}%)")
print(f"              {test_df['date'].min()} to {test_df['date'].max()}")

# Display target distribution per split
print(f"\n📊 Outbreak Rate per Split:")
for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    rate = split_df['outbreak'].mean() * 100
    print(f"   {name}: {rate:.1f}%")

## 10. Export Datasets

In [ ]:
# ============================================
# EXPORT DATASETS
# ============================================

print("="*70)
print("EXPORTING FEATURE ENGINEERED DATASETS")
print("="*70)

# Export full feature dataset
full_features_path = OUTPUT_DIR / 'full_features.csv'
df.to_csv(full_features_path, index=False)
print(f"\n✅ Full features saved: {full_features_path}")
print(f"   Shape: {df.shape}")

# Export train/val/test splits
for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    split_path = MODEL_DIR / f'{split_name}_features.csv'
    split_df.to_csv(split_path, index=False)
    print(f"✅ {split_name.capitalize()} saved: {split_path}")

# Export feature list
feature_list_path = OUTPUT_DIR / 'selected_features.txt'
with open(feature_list_path, 'w') as f:
    f.write('\n'.join(selected_features))
print(f"\n✅ Feature list saved: {feature_list_path}")
print(f"   Total features: {len(selected_features)}")

# Export metadata
metadata = {
    'total_records': len(df),
    'total_features': len(df.columns),
    'selected_features': len(selected_features),
    'train_records': len(train_df),
    'val_records': len(val_df),
    'test_records': len(test_df),
    'date_range': f"{df['date'].min()} to {df['date'].max()}",
    'feature_columns': list(df.columns)
}

import json
metadata_path = OUTPUT_DIR / 'feature_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"✅ Metadata saved: {metadata_path}")

print("\n" + "="*70)
print("FEATURE ENGINEERING COMPLETED!")
print("="*70)
print(f"\n📊 Summary:")
print(f"   Total records: {len(df):,}")
print(f"   Total features: {len(df.columns)}")
print(f"   Selected features: {len(selected_features)}")
print(f"   Train/Val/Test: {len(train_df)}/{len(val_df)}/{len(test_df)}")
print(f"\n📝 Next steps:")
print(f"   1. Run model training: notebooks/03_model_training.ipynb")
print(f"   2. Evaluate model: notebooks/04_model_evaluation.ipynb")
print(f"   3. Deploy model: src/api/predict_api.py")

## 11. Feature Correlation Analysis

In [ ]:
# ============================================
# FEATURE CORRELATION ANALYSIS
# ============================================

print("="*70)
print("FEATURE CORRELATION ANALYSIS")
print("="*70)

# Select top features for correlation
top_features = importance_df.head(15)['feature'].tolist()
corr_features = top_features + ['disease_probability', 'outbreak']

# Calculate correlation
corr_matrix = df[corr_features].corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Correlation heatmap saved: {OUTPUT_DIR / 'correlation_heatmap.png'}")

# Display correlations with target
target_corr = corr_matrix['disease_probability'].sort_values(ascending=False)
print(f"\n📊 Top Correlations with Disease Probability:")
print(target_corr.head(15))

## 12. Feature Distribution Visualization

In [ ]:
# ============================================
# FEATURE DISTRIBUTION VISUALIZATION
# ============================================

print("="*70)
print("FEATURE DISTRIBUTION VISUALIZATION")
print("="*70)

# Select key features to visualize
key_features = [
    'temperature', 'humidity', 'rainfall', 'ndvi', 'ndwi',
    'disease_risk_index', 'stress_index', 'heat_index'
]

# Create distribution plots
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, feature in enumerate(key_features):
    ax = axes[idx]
    
    # Plot by outbreak status
    for outbreak_status in [0, 1]:
        data = df[df['outbreak'] == outbreak_status][feature]
        label = 'No Outbreak' if outbreak_status == 0 else 'Outbreak'
        ax.hist(data, bins=30, alpha=0.6, label=label, density=True)
    
    ax.set_xlabel(feature.replace('_', ' ').title())
    ax.set_ylabel('Density')
    ax.set_title(f'{feature.replace("_", " ").title()} Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Distribution plots saved: {OUTPUT_DIR / 'feature_distributions.png'}")